[![Verify this record with Typed Standards](https://typedstandards.org/badge/typed-standards-verify.svg)](<https://typedstandards.org/verify?url=https%3A%2F%2Fpublish-example.typedstandards.org%2Fbundles%2Fcolab-example.bundle.json>)

| Typed Standards record | |
|---|---|
| Host | `publish-example.typedstandards.org` |
| Capture method | `script-run` |

This cell is a reader affordance and is not authoritative: verification reads the signed record, not this cell, and the verifier shows the record's signer, hash and time.


# A notebook that publishes a record of itself

This notebook runs a small analysis, signs its own `.ipynb` with this example's key, and
publishes the record to `https://publish-example.typedstandards.org` with one call to the
[`typedstandards`](https://pypi.org/project/typedstandards/) package. The repository's
README says what each record proves, and how to run each step.

- Run it in Google Colab, with this notebook's access to the two secrets,
  `TYPEDSTANDARDS_SIGNING_SEED_B64` and `TYPEDSTANDARDS_GITHUB_TOKEN`, turned on in the
  Secrets panel. Use Runtime, Run all.
- `RUN = "first"` publishes `colab-example`. `RUN = "rerun"` publishes
  `colab-example-rerun`, links it to `colab-example` with `revises=`, and then withdraws
  `colab-example`.
- No cell prints either secret. The secrets are read only after the analysis has run, in
  the cell just before signing.
- The record is the notebook as the analysis left it. The cell above is the verifier
  badge for `colab-example`. In the copy it signs, the signing cell replaces it with the
  badge for this run's name.

In [ ]:
# The parameters. RUN is "first" for the first run and "rerun" for the rerun.
RUN = "first"

REPOSITORY = "npstorey/typedstandards-publish-example"
ORIGIN = "https://publish-example.typedstandards.org"
FIRST_NAME = "colab-example"
RERUN_NAME = "colab-example-rerun"
NAME = {"first": FIRST_NAME, "rerun": RERUN_NAME}[RUN]

# The fallback, when Colab cannot return this notebook (the README says when): the path of
# this notebook's .ipynb, downloaded after the analysis ran and uploaded to the Files pane.
NOTEBOOK_FILE = None

In [ ]:
# This notebook's own bytes, which the signing cell signs. The kernel holds no file of the
# notebook: Colab's frontend returns it through get_ipynb, an internal request that Google
# does not document. This cell asks for it now, before any secret is read, and stops the run
# if it cannot have it.
import json
from pathlib import Path

BADGE_START = "[![Verify this record with Typed Standards]"
PARAMETERS_CELL = "# The parameters."
ANALYSIS_CELL = "# The analysis."
SIGNING_CELL = "# The signing cell."


def read_own_notebook():
    """The notebook as Colab's frontend holds it, or as NOTEBOOK_FILE holds it."""
    if NOTEBOOK_FILE:
        where = f"NOTEBOOK_FILE ({NOTEBOOK_FILE})"
        notebook = json.loads(Path(NOTEBOOK_FILE).read_bytes().decode("utf-8"))
    else:
        where = "Colab's get_ipynb"
        try:
            from google.colab import _message
        except ImportError:
            raise RuntimeError("not running in Colab: set NOTEBOOK_FILE to this notebook's .ipynb") from None
        reply = _message.blocking_request("get_ipynb", request="", timeout_sec=60)
        notebook = reply.get("ipynb") if isinstance(reply, dict) else None
        if isinstance(notebook, str):
            notebook = json.loads(notebook)
        if not isinstance(notebook, dict):
            shape = type(reply).__name__ + (f" with keys {sorted(reply)}" if isinstance(reply, dict) else "")
            raise RuntimeError(
                f"{where} answered {shape}, not a notebook: use the fallback in the README (NOTEBOOK_FILE)"
            )
    if notebook.get("nbformat") != 4 or not isinstance(notebook.get("cells"), list):
        raise RuntimeError(f"{where} did not return an nbformat 4 notebook: use the fallback in the README")
    return notebook


def cell_text(cell):
    source = cell.get("source", "")
    return "".join(source) if isinstance(source, list) else source


def joined(value):
    return "".join(value) if isinstance(value, list) else value


def clean_output(output):
    """An output as nbformat defines it, with its metadata dropped."""
    kind = output.get("output_type")
    if kind == "stream":
        return {"name": output["name"], "output_type": kind, "text": joined(output["text"]).splitlines(True)}
    if kind in ("display_data", "execute_result"):
        cleaned = {"data": dict(output.get("data", {})), "metadata": {}, "output_type": kind}
        if kind == "execute_result":
            cleaned["execution_count"] = output.get("execution_count")
        return cleaned
    if kind == "error":
        return {"ename": output["ename"], "evalue": output["evalue"], "output_type": kind,
                "traceback": list(output.get("traceback", []))}
    raise RuntimeError(f"an output of type {kind!r}, which this notebook does not sign")


def code_cell(notebook, start):
    """The code cell whose source starts with ``start``."""
    for cell in notebook["cells"]:
        if cell["cell_type"] == "code" and cell_text(cell).startswith(start):
            return cell
    raise RuntimeError(f"the notebook returned has no cell starting {start!r}")


def signable(notebook):
    """The notebook as it is signed. Every cell keeps its type and source. Code cells above the
    signing cell keep their outputs and execution counts; the signing cell and every later cell
    keep neither. Every cell's metadata is dropped, and of the notebook's only kernelspec and
    language_info are kept: Colab's metadata names the account that ran each cell. The old
    badge cell is dropped; the signing cell inserts this run's."""
    cells, signing = [], False
    for cell in notebook["cells"]:
        kind, text = cell.get("cell_type"), cell_text(cell)
        if kind == "code" and text.startswith(SIGNING_CELL):
            signing = True
        kept = {"cell_type": kind, "metadata": {}, "source": text.splitlines(True)}
        if kind == "code":
            kept["execution_count"] = None if signing else cell.get("execution_count")
            kept["outputs"] = [] if signing else [clean_output(o) for o in cell.get("outputs", [])]
        cells.append(kept)
    if not signing:
        raise RuntimeError("the notebook returned has no signing cell")
    if cells[0]["cell_type"] == "markdown" and cell_text(cells[0]).startswith(BADGE_START):
        cells = cells[1:]
    metadata = notebook.get("metadata", {})
    kept_metadata = {key: metadata[key] for key in ("kernelspec", "language_info") if key in metadata}
    return {"cells": cells, "metadata": kept_metadata, "nbformat": 4, "nbformat_minor": 4}


probe = signable(read_own_notebook())
for start in (PARAMETERS_CELL, ANALYSIS_CELL, SIGNING_CELL):
    code_cell(probe, start)
print(f"This notebook's content is available: {len(probe['cells'])} cells.")

In [ ]:
%pip install --quiet typedstandards==0.2.0

## The analysis

Summary statistics of twelve values made by a fixed formula, so every run prints the same
numbers. It reads no file, no network and no credential.

In [ ]:
# The analysis.
import statistics

values = [40 + (17 * k) % 23 for k in range(1, 13)]
rises = [b - a for a, b in zip(values, values[1:])]
print("values:", values)
print("count:", len(values))
print("mean:", round(statistics.mean(values), 4))
print("median:", statistics.median(values))
print("min, max:", min(values), max(values))
print("largest rise between neighbours:", max(rises))

## Sign and publish

The next cell reads the two secrets from Colab's Secrets panel into this process's
environment, where the `typedstandards` CLI reads the seed and `publish` reads the token. It
prints nothing.

In [ ]:
import os

from google.colab import userdata

# Both secrets are read before either is set, and any value already set is removed first: this
# cell sets nothing unless both reads succeed, and no failure in it can show either value.
NAMES = ("TYPEDSTANDARDS_SIGNING_SEED_B64", "TYPEDSTANDARDS_GITHUB_TOKEN")
for name in NAMES:
    os.environ.pop(name, None)
os.environ.update(zip(NAMES, [userdata.get(name) for name in NAMES]))

In [ ]:
# The signing cell. It signs this notebook as the cells above it left it, with the verifier
# badge for this run's bundle as cell 0, written before signing.
import typedstandards as ts

notebook = signable(read_own_notebook())
if f'RUN = "{RUN}"' not in cell_text(code_cell(notebook, PARAMETERS_CELL)):
    raise RuntimeError(f"the notebook returned does not set RUN = {RUN!r}: it is not this run's notebook")
if not code_cell(notebook, ANALYSIS_CELL)["outputs"]:
    raise RuntimeError("the notebook returned has no outputs for the analysis: it is not this run's notebook")

BUNDLE_URL = f"{ORIGIN}/bundles/{NAME}.bundle.json"
NOTEBOOK_PATH = Path(f"{NAME}.ipynb")
NOTEBOOK_PATH.write_text(json.dumps(notebook, indent=1, sort_keys=True, ensure_ascii=False) + "\n", encoding="utf-8")
ts.badge_cell(BUNDLE_URL, capture_method="script-run", notebook=NOTEBOOK_PATH)

signed = ts.sign(
    {
        "type": "content/analysis/v1",
        "producerProfile": "scripted-recomputation/publish-example",
        "captureMethod": "script-run",
        "prompt": "Summary statistics of twelve values made by a fixed formula, computed in a hosted notebook.",
        "promptVisibility": "full_text",
        "queries": [],
        "dataSources": [],
        "cost": {"model": "none"},
        "skillMetadata": {},
        "trace": {},
        "signer": {"bindingTier": "pseudonymous", "displayName": "typedstandards-publish-example"},
    },
    output_file=NOTEBOOK_PATH,
)
print("signed:", NAME, signed["envelopeHash"])
print("signer:", signed["package"]["signer"]["identifier"])

In [ ]:
# Publish. The rerun first signs a revises node from the first run's record to this one.
import httpx

TITLES = {
    "first": "A hosted notebook's record of itself",
    "rerun": "A hosted notebook's record of itself, rerun",
}


def listed_hash(name):
    """The envelopeHash of the record this host serves as name: its packageHash in the served
    records.json, which a verifier reads too. publish compares it with the repository's main."""
    with httpx.Client(trust_env=False, follow_redirects=False, timeout=30) as client:
        response = client.get(f"{ORIGIN}/records.json")
    response.raise_for_status()
    for record in response.json()["records"]:
        if record["name"] == name:
            return record["packageHash"]
    raise RuntimeError(f"{ORIGIN}/records.json lists no {name}: has its publish run deployed?")


host = ts.GitHubPagesHost(REPOSITORY)
revises = None
if RUN == "rerun":
    revises = ts.attest(
        {
            "type": "attestation/revises/v1",
            "targetNodeId": listed_hash(FIRST_NAME),
            "successorNodeId": signed["envelopeHash"],
            "signer": signed["package"]["signer"],
        }
    )
receipt = ts.publish(signed, host=host, name=NAME, title=TITLES[RUN], revises=revises)
for key in ("name", "commit", "bundle_url", "verify_url", "registry_url", "written"):
    print(f"{key}: {receipt[key]}")

badge = json.loads(signed["package"]["output"])["cells"][0]
print("the badge in cell 0 links to verify_url:", f"](<{receipt['verify_url']}>)" in cell_text(badge))

## The withdrawal (the rerun only)

On the rerun, the next cell withdraws the first run's record, `colab-example`, with a
signed reason. The record stays listed and still verifies; `records.json` marks it
withdrawn with the reason. Run it once: a second run adds a second withdrawal.

In [ ]:
# The withdrawal: on the rerun only, after the rerun is published.
if RUN == "rerun":
    withdrawal = ts.withdraw(
        {
            "targetNodeId": listed_hash(FIRST_NAME),
            "reason": "Replaced by a rerun of the same notebook, published as colab-example-rerun.",
            "signer": signed["package"]["signer"],
        }
    )
    withdrawn = ts.publish_attestation(withdrawal, host=host, name=FIRST_NAME)
    for key in ("name", "commit", "verify_url", "written"):
        print(f"{key}: {withdrawn[key]}")
else:
    print("RUN is first: nothing is withdrawn.")